In [ ]:
import re, pymupdf, chromadb
from bisect import bisect_right
from chromadb.utils import embedding_functions
from langchain_text_splitters import RecursiveCharacterTextSplitter

RUTA_NORMAS = '../data/Primera_y_Segunda_Division.pdf'
RUTA_REGLAS = '../data/Reglas de Juego 2026_27_paginas individuales.pdf'
INDICE_VIEJO = '../data/chroma_index'      # solo lectura: de aquí rescatamos los chunks de Wikipedia
INDICE_NUEVO = '../data/chroma_index_v2'
COLECCION_NUEVA = 'corpus_laliga_v2'

# 500 caracteres ~ 120 tokens: cabe en la ventana del modelo de embeddings (verifícalo en la celda siguiente).
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=60,
                                          separators=["\n\n", "\n", ". ", " "])

def normalizar_espacios(t: str) -> str:
    t = re.sub(r'[ \t]+', ' ', t)
    return re.sub(r'\n{3,}', '\n\n', t).strip()

def limpiar_reglas(t: str) -> str:
    t = re.sub(r'[\ue000-\uf8ff]', '', t)                    # glifo "tofu" de las viñetas (área privada de Unicode)
    t = re.sub(r'[\u2000-\u200f\u2028\u2029\x07]', ' ', t)    # espacios/controles invisibles
    # Quitamos el encabezado repetido de cada página. El capítulo NO se pierde: lo recupera el mapa de más abajo.
    return re.sub(r'Reglas de Juego \d{4}/\d{2}[^\n]*\n?', '', t)

In [ ]:
from sentence_transformers import SentenceTransformer
m = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
print("Ventana del modelo (tokens):", m.max_seq_length)
ejemplo = "Los clubes adscritos al Campeonato Nacional de Liga de Primera División podrán inscribir en su primer equipo, hasta un máximo de tres futbolistas extranjeros no comunitarios pudiendo alinearse simultáneamente."
print("Tokens de una frase de 215 caracteres:", len(m.tokenizer(ejemplo)['input_ids']))
del m

## 1. Normas de Competición 2026/27 → un chunk por apartado, con su artículo

Estrategia: leer línea por línea, descartar el encabezado repetido y el número de página, y abrir un artículo nuevo
cada vez que aparece `ARTÍCULO N. – TÍTULO`. Cada chunk lleva delante `Art. N – TÍTULO` para que, aunque sea un
apartado suelto, el vector sepa de qué trata. `division` evita que las normas de Primera y Segunda (casi idénticas) se mezclen.

In [ ]:
ENCAB = {"CAMPEONATOS NACIONALES DE", "LIGA DE PRIMERA Y SEGUNDA DIVISIÓN"}
ART = re.compile(r'^ARTÍCULO\s+(\d+)\.\s*[–-]\s*(.*)$')
TIT = re.compile(r'^T[ÍI]TULO\s+([IVX]+)$')
DISP = re.compile(r'^DISPOSICIONES (EXTRAORDINARIAS POR FUERZA MAYOR|FINALES)$')

def division(n: int) -> str:
    return 'primera' if 6 <= n <= 12 else 'segunda' if 13 <= n <= 19 else 'ambas'

def armar_cuerpo(lineas: list[str]) -> str:
    # Las líneas del PDF son cortes visuales: solo empieza párrafo nuevo en "1." / "a)" / "-".
    out = []
    for l in lineas:
        if not out or re.match(r'^(\d+\.|[a-f]\)|-)\s', l): out.append(l)
        else: out[-1] += ' ' + l
    return '\n\n'.join(out)

doc = pymupdf.open(RUTA_NORMAS)
print("Página 2 (debe ser el índice):", repr(doc[1].get_text()[:40]))
lineas = [(i + 1, l.strip()) for i in range(2, len(doc))            # saltamos portada e índice
          for l in doc[i].get_text().split('\n')
          if l.strip() and l.strip() not in ENCAB and not l.strip().isdigit()]

arts, actual, k = [], None, 0
while k < len(lineas):
    pag, s = lineas[k]
    if (mt := ART.match(s)):
        nombre = mt.group(2)
        if not nombre.endswith('.'):                                  # título partido en dos líneas
            k += 1; nombre += ' ' + lineas[k][1]
        actual = dict(art=int(mt.group(1)), nombre=nombre.rstrip('.'), pag=pag, texto=[])
        arts.append(actual)
    elif DISP.match(s):                                               # disposiciones finales: no son del Art. 34
        actual = dict(art=0, nombre=s.capitalize(), pag=pag, texto=[]); arts.append(actual)
    elif actual and not TIT.match(s):
        actual['texto'].append(s)
    k += 1

docs_normas = []
for a in arts:
    cab = f"Art. {a['art']} – {a['nombre']}" if a['art'] else a['nombre']
    for ch in splitter.split_text(armar_cuerpo(a['texto'])):
        docs_normas.append(dict(texto=f"{cab}\n{ch.lstrip('. ')}", documento='normas_competicion', seccion=cab,
                                pagina=a['pag'], division=division(a['art']) if a['art'] else 'ambas', articulo=a['art']))

print(len(arts), "secciones;", len(docs_normas), "chunks;", "máx", max(len(d['texto']) for d in docs_normas), "caracteres")
faltan = sorted(set(range(1, 35)) - {a['art'] for a in arts})
print("Artículos que faltan (debe ser []):", faltan)

## 2. Reglas de Juego → capítulo por página con un mapa

Solo 110 de las 276 páginas traen el encabezado `Reglas de Juego 2026/27 | Regla N | Título`. Las demás heredan el capítulo
de la **portada de capítulo**, que está una página antes de la primera página con encabezado. `bisect_right` busca, para
cada página, el último capítulo que empezó en o antes de ella.

In [ ]:
d = pymupdf.open(RUTA_REGLAS)
patron = re.compile(r'Reglas de Juego 2026/27\s*\|\s*([^\n]+)')
encabezados = {}                                                      # página (1-based) -> "Regla 11 | El fuera de juego"
for i, p in enumerate(d):
    mt = patron.search(p.get_text())
    if mt: encabezados[i + 1] = re.sub(r'\s+', ' ', mt.group(1)).strip()
print(len(encabezados), "de", len(d), "páginas traen encabezado")

primeras = {}
for pag, h in encabezados.items(): primeras.setdefault(h, pag)         # primera página de cada capítulo
inicios = sorted((pag - 1, h) for h, pag in primeras.items())          # portada = 1 página antes
paginas = [p for p, _ in inicios]

def seccion_de(n: int) -> str:
    i = bisect_right(paginas, n) - 1
    return inicios[i][1] if i >= 0 else 'Preliminares'

# PRUEBA: en las páginas que SÍ traen encabezado propio, el mapa debe coincidir con él.
malas = [(p, h, seccion_de(p)) for p, h in encabezados.items() if seccion_de(p) != h]
print(len(malas), "inconsistencias de", len(encabezados), "->", malas[:5])
for p in (48, 61, 62, 100, 164, 165):
    print(f"p.{p}: {seccion_de(p)}")

In [ ]:
EXCLUIR = {'Índice', 'Preliminares'}                                    # tablas de contenido: solo meten ruido al buscar
docs_reglas, excluidas = [], 0
for n, pagina in enumerate(d, start=1):
    sec = seccion_de(n)
    texto = normalizar_espacios(limpiar_reglas(pagina.get_text()))
    if sec in EXCLUIR or len(texto) < 80:
        excluidas += 1; continue
    for ch in splitter.split_text(texto):
        if len(ch) >= 80:
            docs_reglas.append(dict(texto=f"[{sec}]\n{ch}", documento='reglas_juego', seccion=sec,
                                    pagina=n, division='', articulo=0))
print(len(docs_reglas), "chunks;", excluidas, "páginas excluidas")
print(docs_reglas[len(docs_reglas)//2]['texto'][:300])

## 3. Wikipedia → rescatamos los chunks ya limpios del índice viejo

Esos dos documentos ya están limpios y validados, así que no repetimos su limpieza: los leemos de la colección vieja
(solo lectura) y los partimos otra vez en trozos de ≤500 caracteres.

In [ ]:
embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(model_name='paraphrase-multilingual-MiniLM-L12-v2')
vieja = chromadb.PersistentClient(path=INDICE_VIEJO).get_collection('corpus_laliga', embedding_function=embedding_fn)
wiki = vieja.get(where={'documento': {'$in': ['primera_division', 'clasico']}}, include=['documents', 'metadatas'])

docs_wiki = []
for t, mt in zip(wiki['documents'], wiki['metadatas']):
    for ch in splitter.split_text(t):
        if len(ch) >= 80:
            docs_wiki.append(dict(texto=ch, documento=mt['documento'], seccion='', pagina=0, division='', articulo=0))
print(len(wiki['documents']), "chunks viejos ->", len(docs_wiki), "chunks nuevos")

## 4. Indexar en una colección nueva (el índice viejo no se toca)

In [ ]:
todos = docs_normas + docs_reglas + docs_wiki
print("Total de chunks:", len(todos))

client = chromadb.PersistentClient(path=INDICE_NUEVO)
try: client.delete_collection(COLECCION_NUEVA)                         # permite re-ejecutar la celda sin error
except Exception: pass
col = client.create_collection(COLECCION_NUEVA, embedding_function=embedding_fn, metadata={"hnsw:space": "cosine"})

CAMPOS = ('documento', 'seccion', 'pagina', 'division', 'articulo')
for i in range(0, len(todos), 500):
    lote = todos[i:i + 500]
    col.add(ids=[f"c{i + j}" for j in range(len(lote))],
            documents=[c['texto'] for c in lote],
            metadatas=[{k: c[k] for k in CAMPOS} for c in lote])
print("Indexados:", col.count())

## 5. Medir la recuperación: ¿la fuente correcta aparece en el top-4?

Cada pregunta lleva el documento esperado y un fragmento del nombre de la sección. **Hit@1** = la fuente correcta es el
primer resultado; **Hit@4** = aparece entre los 4. Esa cifra va al capítulo de resultados de la tesis, y sirve para decidir
con datos (¿hace falta el Libro IV? ¿otro modelo de embeddings?) en lugar de intuición.

In [ ]:
PREGUNTAS = [
    ("¿Cuántos suplentes se pueden usar en un partido?",              'normas_competicion', ['Art. 25']),
    ("¿Cuántas sustituciones puede hacer cada equipo?",               'normas_competicion', ['Art. 25']),
    ("¿Cómo funciona la sustitución por conmoción cerebral?",         'normas_competicion', ['Art. 25']),
    ("¿Cuántos equipos descienden de Primera División?",              'normas_competicion', ['Art. 8']),
    ("¿Cuántos puntos se dan por una victoria?",                      'normas_competicion', ['Art. 7']),
    ("¿Cuántos equipos participan en Primera División?",              'normas_competicion', ['Art. 6']),
    ("¿Cómo se desempata si dos equipos terminan con los mismos puntos?", 'normas_competicion', ['Art. 10', 'Art. 17']),
    ("¿Cuántas amonestaciones llevan a una suspensión?",              'normas_competicion', ['Art. 34']),
    ("¿Cuántos futbolistas extranjeros no comunitarios puede inscribir un club?", 'normas_competicion', ['Art. 22']),
    ("¿Cuándo abre el periodo de inscripción de licencias de invierno?", 'normas_competicion', ['Art. 23']),
    ("¿En qué horarios se juegan los partidos del sábado?",           'normas_competicion', ['Art. 31']),
    ("¿Qué es el fuera de juego?",                                    'reglas_juego', ['Regla 11']),
    ("¿Cuánto dura un partido?",                                      'reglas_juego', ['Regla 7']),
    ("¿Cómo se ejecuta un penal?",                                    'reglas_juego', ['Regla 14', 'Regla 12']),
    ("¿Qué es un saque de esquina?",                                  'reglas_juego', ['Regla 17']),
    ("¿Para qué sirve el VAR?",                                       'reglas_juego', ['VAR']),
    ("¿Qué es El Clásico entre el Barcelona y el Real Madrid?",       'clasico', ['']),
]

def acierta(meta, doc_esperado, subcadenas):
    return meta['documento'] == doc_esperado and any(s in meta['seccion'] for s in subcadenas)

hit1 = hit4 = 0
for q, doc_esp, subs in PREGUNTAS:
    metas = col.query(query_texts=[q], n_results=4)['metadatas'][0]
    pos = next((i for i, mt in enumerate(metas) if acierta(mt, doc_esp, subs)), None)
    hit1 += pos == 0; hit4 += pos is not None
    marca = '✔' if pos is not None else '✘'
    print(f"{marca} pos={pos} | {q}")
    if pos is None: print("     obtuve:", [(mt['documento'], mt['seccion'][:30]) for mt in metas])
print(f"\nHit@1: {hit1}/{len(PREGUNTAS)}   Hit@4: {hit4}/{len(PREGUNTAS)}")